In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, GRU,
    Conv1D, MaxPooling1D, BatchNormalization, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("GPU:", tf.config.list_physical_devices('GPU'))


In [ ]:
FOLDER_PATH = "/content/drive/MyDrive/project_mid"

excel_files = glob.glob(os.path.join(FOLDER_PATH, "*.xlsx"))
print(f"Files found: {len(excel_files)}")

if len(excel_files) == 0:
    raise Exception("No Excel files found. Check FOLDER_PATH.")

all_dataframes = []
for file in excel_files:
    print("Loading:", os.path.basename(file))
    df = pd.read_excel(file)
    all_dataframes.append(df)

combined_df = pd.concat(all_dataframes, ignore_index=True)
print("Combined shape:", combined_df.shape)
combined_df.head()


In [ ]:
# ── Clean columns ──
combined_df.columns = combined_df.columns.str.strip()
combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

# ── Datetime conversion ──
for col in ['From Date', 'To Date']:
    if col in combined_df.columns:
        combined_df[col] = pd.to_datetime(
            combined_df[col].astype(str).str.strip(),
            errors='coerce', format='mixed'
        )

if 'From Date' in combined_df.columns:
    combined_df = combined_df.dropna(subset=['From Date'])
    combined_df = combined_df.sort_values('From Date')

combined_df = combined_df.drop_duplicates().reset_index(drop=True)

# ── Convert to numeric ──
date_cols = ['From Date', 'To Date']
for col in combined_df.columns:
    if col not in date_cols:
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

# ── Handle missing values ──
numeric_cols = combined_df.select_dtypes(include=[np.number]).columns
combined_df[numeric_cols] = (
    combined_df[numeric_cols]
    .interpolate(method='linear', limit_direction='both')
    .ffill().bfill().fillna(0)
)

# ── Feature engineering ──
if 'From Date' in combined_df.columns:
    combined_df['hour']    = combined_df['From Date'].dt.hour
    combined_df['day']     = combined_df['From Date'].dt.day
    combined_df['month']   = combined_df['From Date'].dt.month
    combined_df['weekday'] = combined_df['From Date'].dt.weekday

# ── Feature selection ──
TARGET = 'PM2.5'

possible_features = [
    'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'SO2', 'CO', 'Ozone',
    'RH', 'WS', 'WD', 'BP', 'AT', 'RF', 'TOT-RF',
    'Benzene', 'Toluene', 'Eth-Benzene', 'MP-Xylene', 'O-Xylene',
    'SR', 'Temp', 'VWS', 'hour', 'day', 'month', 'weekday'
]
features = [c for c in possible_features if c in combined_df.columns]

combined_df = combined_df.dropna(subset=[TARGET])
X = combined_df[features]
y = combined_df[TARGET]
print(f"Features: {X.shape} | Target: {y.shape}")

# ── Normalization ──
feature_scaler = MinMaxScaler()
target_scaler  = MinMaxScaler()
X_scaled = feature_scaler.fit_transform(X)
y_scaled = target_scaler.fit_transform(y.values.reshape(-1, 1))

# ── Sequence generation (24-hour sliding window) ──
SEQ_LEN = 24

def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i - seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_scaled, y_scaled, SEQ_LEN)
print(f"X_seq: {X_seq.shape} | y_seq: {y_seq.shape}")

# ── Train / Val / Test split (70 / 15 / 15) ──
n = len(X_seq)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X_seq[:train_end],  y_seq[:train_end]
X_val,   y_val   = X_seq[train_end:val_end], y_seq[train_end:val_end]
X_test,  y_test  = X_seq[val_end:],    y_seq[val_end:]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# ── Save processed dataset ──
combined_df.to_csv(os.path.join(FOLDER_PATH, "processed_dataset.csv"), index=False)
print("Preprocessed data saved.")


In [ ]:
# ── Evaluation function ──
def evaluate_model(y_true, y_pred, name):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"\n{name}  →  MAE:{mae:.4f}  RMSE:{rmse:.4f}  R2:{r2:.4f}  MAPE:{mape:.2f}")
    return [name, mae, rmse, r2, mape]

# ── Shared callbacks ──
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0)
]

# ── Flatten for sklearn models ──
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0],  -1)


In [ ]:
# ── Linear Regression ──
lr_model = LinearRegression(n_jobs=-1)
lr_model.fit(X_train_flat, y_train)
lr_pred    = lr_model.predict(X_test_flat)
lr_results = evaluate_model(y_test, lr_pred, "Linear Regression")

# ── Random Forest ──
rf_model = RandomForestRegressor(
    n_estimators=150, max_depth=20,
    min_samples_split=5, min_samples_leaf=2,
    max_features='sqrt', n_jobs=-1, random_state=42
)
rf_model.fit(X_train_flat, y_train.ravel())
rf_pred    = rf_model.predict(X_test_flat)
rf_results = evaluate_model(y_test, rf_pred, "Random Forest")


In [ ]:
INPUT_SHAPE = (X_train.shape[1], X_train.shape[2])
FIT_ARGS = dict(
    epochs=30, batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=callbacks, verbose=1
)

# ── LSTM ──
lstm_model = Sequential([
    Input(shape=INPUT_SHAPE),
    LSTM(64, return_sequences=True), Dropout(0.2),
    LSTM(32),                        Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')
lstm_history = lstm_model.fit(X_train, y_train, **FIT_ARGS)
lstm_pred    = lstm_model.predict(X_test)
lstm_results = evaluate_model(y_test, lstm_pred, "LSTM")

# ── GRU ──
gru_model = Sequential([
    Input(shape=INPUT_SHAPE),
    GRU(64, return_sequences=True), Dropout(0.2),
    GRU(32),                        Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
gru_model.compile(optimizer='adam', loss='mse')
gru_history = gru_model.fit(X_train, y_train, **FIT_ARGS)
gru_pred    = gru_model.predict(X_test)
gru_results = evaluate_model(y_test, gru_pred, "GRU")

# ── CNN-BiLSTM ──
cnn_bilstm_model = Sequential([
    Input(shape=INPUT_SHAPE),
    Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Bidirectional(LSTM(48, return_sequences=True, recurrent_dropout=0.1)), Dropout(0.2),
    Bidirectional(LSTM(24, return_sequences=False, recurrent_dropout=0.1)), Dropout(0.2),
    Dense(24, activation='relu'),
    Dense(8,  activation='relu'),
    Dense(1)
])
cnn_bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse'
)
cnn_bilstm_history = cnn_bilstm_model.fit(X_train, y_train, **FIT_ARGS)
cnn_bilstm_pred    = cnn_bilstm_model.predict(X_test)
cnn_bilstm_results = evaluate_model(y_test, cnn_bilstm_pred, "CNN-BiLSTM")


In [ ]:
# ── Comparison table ──
results_df = pd.DataFrame(
    [lr_results, rf_results, lstm_results, gru_results, cnn_bilstm_results],
    columns=['Model', 'MAE', 'RMSE', 'R2', 'MAPE']
)
print(results_df.to_string(index=False))

best = results_df.loc[results_df['R2'].idxmax()]
print(f"\nBest model (R2): {best['Model']}  R2={best['R2']:.4f}")
results_df.to_csv("final_model_comparison.csv", index=False)

# ── Bar plots (MAE, RMSE, R2, MAPE) ──
for metric in ['MAE', 'RMSE', 'R2', 'MAPE']:
    plt.figure(figsize=(9, 4))
    plt.bar(results_df['Model'], results_df[metric], color='steelblue')
    plt.title(f'{metric} Comparison')
    plt.ylabel(metric)
    plt.grid(axis='y')
    plt.tight_layout()
    plt.savefig(f"{metric}_comparison.png", dpi=150)
    plt.show()

# ── Prediction comparison ──
plt.figure(figsize=(16, 5))
plt.plot(y_test[:300], label='Actual', linewidth=1.5)
for pred, label in [
    (lr_pred,         'Linear Regression'),
    (rf_pred,         'Random Forest'),
    (lstm_pred,       'LSTM'),
    (gru_pred,        'GRU'),
    (cnn_bilstm_pred, 'CNN-BiLSTM'),
]:
    plt.plot(pred[:300], label=label, linewidth=1)
plt.title('Model Prediction Comparison')
plt.xlabel('Samples'); plt.ylabel('PM2.5 (scaled)')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig("prediction_comparison.png", dpi=150)
plt.show()

# ── CNN-BiLSTM loss curve ──
plt.figure(figsize=(9, 4))
plt.plot(cnn_bilstm_history.history['loss'],     label='Train Loss')
plt.plot(cnn_bilstm_history.history['val_loss'], label='Val Loss')
plt.title('CNN-BiLSTM Training Curve')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()


In [ ]:
lstm_model.save("lstm_model.h5")
gru_model.save("gru_model.h5")
cnn_bilstm_model.save("cnn_bilstm_model.h5")
print("Models saved.")
